# Lab 3: Vision Pipeline & Profiling

**Author:** Dr. Le Viet Duc, Hai-Long Nguyen

**Copyright:** © 2026 University of Twente. All rights reserved.

**License & Usage Terms:**
This notebook and its contents are provided strictly for educational purposes within the context of the **Intelligent Computing for Embedded Systems** course.

- **Permitted:** Students enrolled in this course may run, modify, and save a personal copy of this notebook for their own educational use and assignment submission.

- **Prohibited:** You may not distribute, publish, or share this material publicly. This includes, but is not limited to, uploading to public GitHub repositories, course-sharing websites (e.g., CourseHero, Chegg), or public Google Drive folders.

Unauthorized distribution constitutes an academic integrity violation and a breach of intellectual property rights.

---


**Duration:** 180 minutes  
**Follows:** Lecture 3 (Attention, Transformers & Edge Deployment)  
**Platform:** RPi 5 + Hailo AI HAT+ + Pi Camera Module 3  
**Group size:** 5

> **📊 ASSIGNMENT NOTE:** This lab produces the most critical data for the assignment.
> The per-stage profiling breakdown and 4-model comparison are directly used in
> Part B of the assignment. Record everything carefully.

---


## Pre-Lab (at home)

- Complete all previous LAB 2a, 2b.
- For the one who has not install Hailo Dataflow Compiler, please follow the guideline `Guideline-HailoDataflowCompiler-Setup.pdf` to set up the compiler on your local machine. This allows you to compile models to `.hef` format at home and save time during the lab session.
- Download YOLO26n weights and save it to your local machine.

### 👥 Sub-group split for Lab 3 / 3b

To keep both the workstation and the Raspberry Pi busy in parallel, the lab group is split into **two sub-groups**:

| Sub-group | Notebook | Hardware | Deliverable |
|---|---|---|---|
| **A — Model builders** | `lab3b_flowers102_multimodel_teacher.ipynb` | Workstation (GPU) | Train 4 classifiers on Oxford Flowers-102, export ONNX, compile **`.hef`** (Part E2), build **`flowers102_test.npz`** (Part F) |
| **B — Edge benchmarkers** | `lab3_vision_pipeline_profiling.ipynb` (this notebook) | Raspberry Pi 5 + Hailo-10H | Run real-time YOLO26n detection (Parts A–B) and **accuracy evaluation of all 4 `.hef` classifiers** on the Hailo-10H (Part C1) |

Hand-off: sub-group A copies the four `.hef` files plus `flowers102_test.npz` to the Pi for sub-group B to evaluate.

---


## Part A: Real-Time Object Detection (40 min)

### A0. YOLO 26n convert to `.hef`

While a student try to convert the downloaded YOLO26n weight to `.hef` format using Hailo's tools, the other student can start with the next step to set up the X11 forwarding to run the real-time detection demo on the Pi. 

Checkout guideline `Guideline-X11-forwarding.pdf` and section **A1** to know what to do after having the .hef file and X11 forwarding set up.

First, you need to download **YOLO26n** weight (detection task) from [Ultralytics YOLO Detection](https://docs.ultralytics.com/tasks/detect). 

Then, use the bellow code cell to convert the downloaded YOLO26n weight to `.onnx` format.

In [ ]:
from ultralytics import YOLO


imgsz = 640
batch = 1
opset = 11


model_weight = <path_to_yolo26n_weights>  # e.g., "yolo26n.pt"
model = YOLO(model_weight)
output_onnx = <path_to_output_onnx>  # e.g., "yolo26n.onnx"


exported = model.export(
    format="onnx",
    imgsz=imgsz,
    batch=batch,
    opset=opset,
    simplify=False,
    dynamic=True,
    nms=False,
    half=False,
    device="cpu",
)

from pathlib import Path
import shutil

# Path processing for the exported ONNX file
exported = Path(exported)
if output_onnx is not None:
    output_onnx = Path(output_onnx)
    output_onnx.parent.mkdir(parents=True, exist_ok=True)
    if exported.resolve() != output_onnx.resolve():
        shutil.move(str(exported), str(output_onnx))
    exported = output_onnx
print(f"Done. ONNX saved to: {exported}")

After running the code cell, you will get a `yolo26n.onnx` file.

Next, use the `convert_hef_file.py` file to convert the `yolo26n.onnx` file to `yolo26n.hef` file. 

You need to reference the guideline `Guideline-HEF-conversion.pdf` to run the conversion with **calibration dataset.**

* Because **group A** also need calibration dataset to run their conversion but all models in Lab 3b need data with size 224x224, so make another calibration dataset with 512 images sized 224x224 (reference how to make the calibration dataset in guideline `Guideline-HEF-conversion.pdf`).

### A1. YOLO26n Pipeline

After having the `yolo26n.hef` file, the inference pipeline is ready to run on the Hailo-10H.

**Before doing this part**, you need to read guideline `Guideline-X11Forwarding-Setup.pdf` to set up X11 forwarding on your local machine. This allows you to run graphical applications (like `hailo_viz`) from the Raspberry Pi and display them on your local machine.

Run your model on CPU of Raspberry Pi 5 with `.onnx` file:
```bash
python3 pipeline_onnx_cpu.py --model yolo26n.onnx --conf-threshold 0.25 --source picamera --display
```


**Run your model on Hailo AI HAT with `.hef` file:**
```bash
python3 pipeline_hef_npu.py --model yolo26n.hef --conf-threshold 0.25 --iou-threshold 0.4 --source picamera --display
```
Point camera at objects. Record observed FPS.

1. **Compare the two.** Which is faster? By how much?

2. **Modify the conf-threshold and iou-threshold.** How does it affect the FPS and detection quality?


**Run headless mode to get FPS without display overhead:**
```bash
python3 pipeline_onnx_cpu.py --model yolo26n.onnx --conf-threshold 0.25 --source picamera --no-display --iterations 200

python3 pipeline_hef_npu.py --model yolo26n.hef --source picamera --no-display --iterations 200 

```
Record headless FPS. 

Is there any difference from the FPS with display option? If so, why?


### A2. Detection Under Different Conditions

A student operates camera, while another records.

**What to measure.** For each object, hold it in front of the Pi Camera under the three test conditions below and read the **confidence score** that appears next to the bounding-box label in the display window (e.g., the `0.87` in `bottle 0.87`). Record it as a percentage (`0.87` → `87%`). If the model fails to draw a box for the object, write `—` (no detection).

- **Normal (conf%)** — object fully visible, room lighting, ~50 cm from the camera. Baseline confidence.

- **Occluded 50% (conf%)** — cover roughly half of the object with your hand or another item so only ~50% is visible to the camera. Tests robustness to partial occlusion.

- **Low Light (conf%)** — dim the room lights (or shade the object) so the scene is clearly darker, but the object is still visible to a human. Tests robustness to poor illumination.

- **Correct?** — write `Y` if the predicted class label matches the true object (e.g., `bottle` for a bottle), `N` if the label is wrong, `—` if nothing was detected.

Report the **median over ~5 seconds** of stable framing per cell (confidence fluctuates frame-to-frame), not a single spike.

**Table**: record confidence scores and correctness for each object under each condition 

(run on CPU with `.onnx`):

| Object | Normal (conf%) | Occluded 50% (conf%) | Low Light (conf%) | Correct? |
|--------|---------------|---------------------|------------------|----------|
| Bottle | | | | |
| Keyboard | | | | |
| Phone | | | | |
| Cup | | | | |
| Book | | | | |

**Table**: record confidence scores and correctness for each object under each condition 

(run on NPU Hailo 10H with `.hef`):

| Object | Normal (conf%) | Occluded 50% (conf%) | Low Light (conf%) | Correct? |
|--------|---------------|---------------------|------------------|----------|
| Bottle | | | | |
| Keyboard | | | | |
| Phone | | | | |
| Cup | | | | |
| Book | | | | |


## Part B: Per-Stage Profiling (60 min)


### B1. Latency Breakdown

**Run 1:** Run benchmark with per-stage profiling enabled to get the latency breakdown of each pipeline stage. Use a large number of iterations to get stable percentiles.
```bash
python3 benchmark.py --model yolo26n.hef --source picamera --iterations 1000 --warmup 100 --profile
```
---

**Run 2:** Run independently with `--iterations 500` to check reproducibility. 

**Document records** (critical for assignment):

**What to copy into the table.** `benchmark.py --profile` prints a block titled `=== B1 per-stage breakdown ===` with one row per stage and three percentile columns. Copy those numbers straight in. All latencies are in **milliseconds**.

- **p50 (median)** — half of frames are faster than this, half slower. The single best number for "what does the pipeline usually take?"
- **p95** — only 5% of iterations were slower than this. Tells you about the long tail.
- **p99** — only 1% of iterations were slower. The worst-case budget for a real-time guarantee has to account for this.
- **Runs On** — which hardware unit executes the stage. `ISP` = the image signal processor on the Pi Camera Module 3. `CPU` = the Pi 5's Cortex-A76 cores. `PCIe` = the bus between the Pi and the Hailo HAT+. `Hailo` = the NPU silicon itself.
- **Total** — end-to-end wall-clock for one full frame (capture → display-ready detections). Not exactly the sum of stage p50s, because tail events on different stages don't always coincide.
- **Effective FPS** — `1000 / Total p50`. The throughput you'd observe in steady state.

> *Why percentiles instead of mean?* On a shared OS, occasional pauses (kernel preemption, GC, IRQs, thermal throttling) skew the mean. Percentiles separate "typical" from "tail" behavior — which is what matters for real-time work.

> *Note on the PCIe rows.* HailoRT's `InferVStreams` API exposes `Host→NPU + NPU compute + NPU→Host` as a single blocking call, so the benchmark reports those PCIe rows as `—` and rolls the cost into the `NPU inference` row. The end-to-end cost is captured; only the internal split is hidden.

---

| Pipeline Stage | p50 (ms) | p95 (ms) | p99 (ms) | Runs On |
|----------------|---------|---------|---------|---------|
| Camera capture | | | | ISP |
| Resize + normalize | | | | CPU |
| Host→NPU transfer | | | | PCIe |
| NPU inference | | | | Hailo |
| NPU→Host transfer | | | | PCIe |
| NMS post-processing | | | | CPU |
| **Total** | | | | |
| **Effective FPS** | | | | |

**Reproducibility check.** Run 2 (`--iterations 500`) should agree with Run 1 within a few percent on every row. If a row drifts more than that, something else was competing for the same hardware unit (a background process on the CPU, throttling, etc.) — investigate and re-run.

**Run 3:** Run independently with `--iterations 200` and `--no-warmup`/`--warmup 100` to check reproducibility. 

**Warm-up vs no-warm-up.** Compare the p50 from `--no-warmup --iterations 200` against the warmed-up p50 and record both.

1. Which p50 is higher ? `--no-warmup` or `--warmup 100` ? 

2. Explain why there is a difference. 


### B2. Bottleneck Analysis


1. Theoretical NPU time: FLOPs / peak_TOPS. Calculate the theoretical NPU time for YOLO26n and compare against the actual NPU inference p50 from the benchmark.

In [ ]:
from ultralytics import YOLO

# Load YOLO26n (uses already-loaded `model` if available, otherwise loads from weights)
model = YOLO("yolo26n.pt")

from ultralytics.utils.torch_utils import get_flops

# Run this on Raspberry Pi to get the 

# Recompute GFLOPs at the actual inference resolution
# get_flops() actually return number of multiply-adds, 
# so we need to multiply by 2 to get FLOPs.
gflops_pi = ...
print(f"YOLO26n @ 640x640: {gflops_pi:.2f} GFLOPs")

# Fill the peak TOPS of Hailo-10H (from the datasheet) to compute the theoretical NPU time.
peak_TOPS = ...

# Theoretical NPU time (ms) = GFLOPs / (peak_TOPS * 1000) * 1000
theoretical_npu_time_ms = ...
print(f"Theoretical NPU time for YOLO26n @ 640x640: {theoretical_npu_time_ms:.2f} ms")

actual_npu_time_ms = ...
print(f"Actual NPU time for YOLO26n @ 640x640: {actual_npu_time_ms:.2f} ms")

2. How many percent that the post-processing stage takes ?


3. If NPU were infinitely fast (0 ms), what's the max FPS from remaining stages? How can you calculate it from the benchmark table ?

*Support command:* Run this command on Raspberry Pi to get the input size of model:
```bash
hailortcli parse-hef yolo26n.hef | grep -A2 "input"
```

### B3. Thermal Analysis

You run the experiment **twice**: once offloading inference to the Hailo NPU (`.hef`), once running everything on the Pi 5's CPU (`.onnx`). Comparing the two shows how each backend heats the board and whether thermal throttling costs you throughput.

`thermal_logger.py` samples three things every `--interval` seconds and writes one CSV row per sample:

- **CPU temperature** — read from `vcgencmd measure_temp` (with a `/sys/class/thermal/thermal_zone0/temp` fallback).

- **Hailo NPU temperature** — read through the **HailoRT Python API**, `Device.control.get_chip_temperature()`, reporting the hotter of the two on-die sensors, `max(TS0, TS1)`. (The older `hailortcli fw-control identify` text does **not** carry temperature on current firmware, so don't rely on it.) On the CPU/ONNX run the NPU sits idle, so this column just reflects its idle temperature.

- **Pipeline FPS** — read from the JSON status file each pipeline writes to `/tmp/pipeline_status.json` every ~0.5 s. The two terminals stay in sync through this file with no IPC plumbing on your side. If the file goes stale (no update within `--stale-after`, default 15 s), FPS is logged blank.

CSV columns: `timestamp_iso, elapsed_s, cpu_temp_c, hailo_temp_c, fps`. When each run finishes, the logger prints the Table 2 metrics under `=== Summary (Table 2 inputs) ===`.

#### Run 1 — NPU (`.hef`) on the Hailo-10H
```bash
# Terminal 1: continuous inference on the NPU
python3 pipeline_hef_npu.py --model yolo26n.hef --source picamera --no-display --duration 300

# Terminal 2: temperature + FPS logging every 10 seconds
python3 thermal_logger.py --duration 300 --interval 10 --output thermal_log_hef.csv
```

#### Run 2 — CPU (`.onnx`) on the Pi 5
Repeat the experiment with the CPU pipeline, so you can compare how much the board heats up when the Cortex-A76 cores do all the inference instead of offloading it to the NPU.
```bash
# Terminal 1: continuous inference on the CPU
python3 pipeline_onnx_cpu.py --model yolo26n.onnx --source picamera --no-display --duration 300

# Terminal 2: temperature + FPS logging every 10 seconds
python3 thermal_logger.py --duration 300 --interval 10 --output thermal_log_onnx.csv
```

After each run, plot the thermal log: x-axis = time (s), y-axis = temperature (°C) and FPS. If you can, overlay both runs on the same axes to compare CPU-only vs NPU-offloaded heating.

**Table 1: Record CPU temp, Hailo temp, and FPS at each time point.**

Open each CSV. For each row of the tables below, find the line whose `elapsed_s` value is closest to the target time and copy `cpu_temp_c`, `hailo_temp_c`, `fps`. The `0s (start)` row uses the first sample of the CSV.

*Run 1 — NPU (`.hef`), from `thermal_log_hef.csv`:*

| Time | CPU Temp (°C) | Hailo Temp (°C) | FPS |
|------|-------------|----------------|-----|
| 0s (start) | | | |
| 60s | | | |
| 120s | | | |
| 180s | | | |
| 240s | | | |
| 300s (end) | | | |

*Run 2 — CPU (`.onnx`), from `thermal_log_onnx.csv` (NPU idle, so its temp column should stay roughly flat):*

| Time | CPU Temp (°C) | Hailo Temp (°C) | FPS |
|------|-------------|----------------|-----|
| 0s (start) | | | |
| 60s | | | |
| 120s | | | |
| 180s | | | |
| 240s | | | |
| 300s (end) | | | |

**Table 2: Summarize thermal impact on performance.**

The logger prints all six values under `=== Summary (Table 2 inputs) ===` when each run finishes. Copy them straight in — one column per backend.

- **Peak FPS (first 30s)** — max FPS observed in the first 30 seconds, before the silicon has heated up.
- **Sustained FPS (last 60s)** — mean FPS over the final minute, after temperatures have stabilized.
- **FPS drop (%)** — `(peak − sustained) / peak × 100`. Quantifies how much throughput thermal throttling cost you.
- **Max CPU / Hailo temperature** — the highest value observed in the corresponding column of the CSV.
- **Time to thermal steady-state** — the elapsed time at which CPU temperature first reaches within 2 °C of its eventual maximum (the logger's defensible proxy for "temperature has stopped rising meaningfully").

| Metric | NPU (`.hef`) | CPU (`.onnx`) |
|--------|--------------|---------------|
| Peak FPS (first 30s) | | |
| Sustained FPS (last 60s) | | |
| FPS drop (%) | | |
| Max CPU temperature | | |
| Max Hailo temperature | | |
| Time to thermal steady-state | | |

Does FPS drop correlate with temperature rise?

At what temperature does throttling begin?

Which backend runs hotter and which throttles more — and why? (Think about where the compute happens in each run.)

---


## Part C: Multi-Model Comparison (55 min)


### C1. Benchmark Four Models

**Before running the benchmark**, make sure you already have on the Raspberry Pi:

- Four classifier HEFs from **Lab 3b Part E2** (built and shipped by sub-group A), placed under `week-4/hef_models/`:
  - `mobilenet_v2_flower102.hef`
  - `efficientnet_b0_flowers102.hef`
  - `resnet18_flowers102.hef`
  - `yolo26n_cls_flowers102.hef`
- The evaluation dataset `flowers102_test.npz` from **Lab 3b Part F**, copied next to the HEFs.



#### Step 1 — Latency / FPS benchmark (all 4 classifiers + YOLO26n)

```bash
cd ~/IES-LABs-1/week-4


python3 benchmark.py \
    --model <path-to-hef-file> \
    --iterations 1000 --warmup 100 \
    --output-csv c1_summary.csv
```

Passing `--output-csv c1_summary.csv` to every run aggregates one row per model into a single CSV you can paste in directly.


#### Step 2 — Top-1 accuracy on the Hailo-10H (Oxford Flowers-102)

`evaluate_accuracy.py` loads `flowers102_test.npz`, runs every test image through the `.hef` on the NPU, and reports **top-1 / top-5 accuracy** plus throughput. See [`evaluate_accuracy.py`](evaluate_accuracy.py) for the full CLI.

```bash
# One row per model, appended to c1_accuracy.csv

python3 evaluate_accuracy.py \
    --model hef_models/${m}.hef \
    --test-data flowers102_test.npz \
    --output-csv c1_accuracy.csv

```

The script:
1. Opens the `.hef`, reads its input H×W, and resizes the cached test images if needed.
2. Runs one inference per image on the NPU (`uint8` NHWC in, `float32` logits out).
3. Counts top-1 and top-K hits, then prints a summary and appends one row to `c1_accuracy.csv`.



**Critical for assignment:**

**Table: Record model size, latency, and accuracy for all 4 models. This is the main data for Part B of the assignment.**

For each row: take **.hef (MB)**, **p50**, **p95**, and **FPS** from `benchmark.py`'s `=== C1 summary ===` block (or from `c1_summary.csv`). Take **Top-1 / Top-5 Accuracy** from `c1_accuracy.csv` (produced by `evaluate_accuracy.py`). Look up **Params (M)** from each model's documentation — it isn't recoverable from the compiled `.hef`.

- **Task** — what the model outputs. `Classification` = single label per image. `Detection` = bounding boxes + labels.
- **Params (M)** — total trainable parameters of the original (un-compiled) model, in millions. Take this from the count printed in Lab 3b Part B1.
- **.hef (MB)** — size of the compiled Hailo binary on disk. Printed by the benchmark.
- **FPS** — sustained throughput at `--iterations 1000`. Equal to `1000 / p50`.
- **p50 / p95 (ms)** — median and tail latency from the benchmark.
- **Top-1 Acc (%)** — top-1 classification accuracy on the Flowers-102 test set, INT8 on Hailo-10H.

| Model | Task | Params (M) | .hef (MB) | FPS | p50 (ms) | p95 (ms) | Top-1 Acc (%) |
|-------|------|-----------|----------|-----|---------|---------|---------------|
| MobileNetV2 | Classification | | | | | | |
| EfficientNet-B0 | Classification | | | | | | |
| ResNet18 | Classification | | | | | | |
| YOLO26n-cls | Classification | | | | | | |
| YOLO26n | Detection | | | | | | — (qualitative, see Part A2) |

Compare the Top-1 column against the **PyTorch fp32** numbers reported in Lab 3b Part D. The gap is the cost of INT8 quantization on the Hailo-10H. 

**Which solution can be used to reduce the accuracy drop caused by quantization ?**


### C2. Roofline from Measurements

The roofline model tells you whether each model is **compute-bound** (limited by NPU FLOPs/s) or **memory-bound** (limited by PCIe / on-chip bandwidth). For each model, plug the C1 p50 latency into the formulas below.

For each model:
1. **Achieved TOPS** = `FLOPs / measured_latency_seconds`. Use the p50 latency from C1 (convert ms → s). FLOPs comes from the model's documentation (Hailo Model Zoo or paper).
2. **NPU utilization** = `achieved / 20 TOPS`. The Hailo-10H peak is 20 TOPS. Models hitting > 50% are well-matched to the silicon; models below ~10% are throttled by something other than compute (usually memory bandwidth or post-processing).
3. **Operational intensity** `I` = `FLOPs / bytes_read`. Approximate `bytes_read` as the sum of input bytes (`H × W × C × precision_bytes`) and weight bytes (≈ `params × precision_bytes`). High `I` → compute-bound. Low `I` → memory-bound.

Create a roofline plot (matplotlib) with all 4 models — x-axis = `I` (FLOPs/byte, log scale), y-axis = achieved TOPS (log scale), with the peak-compute roof (horizontal line at 20 TOPS) and the peak-bandwidth roof (sloped line at `bandwidth × I`) drawn. Each model is one point; its position relative to the two roofs tells you which one is binding.

Save the plot and record the utilization numbers.

In [ ]:
# Calculate FLOPs

# Getting measured latency second (from table)

# Calculate achieved TOPS = FLOPs / measured_latency_seconds

# Calculate NPU utilization = achieved / 13 TOPS

# Calculate total parameters

# Calculate model size in MB

# Calculate operational intensity I

In [ ]:
# Create roofline plot (matplotlib) with all 4 models.


## Part D: Reflection + Save

### Use-Case Design (15 min) — All

Scenario: **Retail store** — count customers, estimate dwell time near displays.
1. Which model would you choose?
2. What FPS do you need?
3. Would thermal throttling be a problem for continuous operation?
4. Would you apply compression from Lab 2b?

**Documenter:** Record group's recommendation.


### Save for Assignment (10 min)

1. **Documenter:** All tables, thermal plot, and roofline plot in shared group document
2. **Co-Driver:** Save roofline plot code and thermal CSV
3. **Driver:** Shut down RPi
4. **All:** Verify shared document is complete — this is the last data-heavy lab
